# 第 11 章　日時と時間の変化

一日の合計と時間別の利用を見比べ，どの時間帯に備えるかを考えます。

このノートの前半は，本文の説明と一緒に実行します。各場面で先に短い問いを読み，講師の案内に合わせてセルを動かし，出力の見所を確認します。後半は約20分の実習です。前半のコードは動く見本として用意してあり，作図コードをすべて書き写す必要はありません。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV は，手元にファイルがなければ公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

file_name = "bike-sharing-hourly.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

hourly = pd.read_csv(csv_path)
hourly[["dteday", "hr", "cnt", "workingday"]].head(3)

## 本文と一緒に実行

### 1. 日合計が近い二日を読む

6月7日と9日は日合計がほぼ同じでした。8時台の貸出件数も近いでしょうか。まず日付を日時型へ直し，二日の合計と時間別の図を作ります。

In [ ]:
print(type(hourly["dteday"].iloc[0]).__name__)
hourly["dteday"] = pd.to_datetime(hourly["dteday"], format="%Y-%m-%d")
print(type(hourly["dteday"].iloc[0]).__name__)
thursday = pd.Timestamp("2012-06-07")
saturday = pd.Timestamp("2012-06-09")
sample_hours = hourly.loc[hourly["dteday"].isin([thursday, saturday])]
display(sample_hours.groupby("dteday")["cnt"].sum())
weekday_color, weekend_color = "#195b8a", "#a7421f"
plt.rcParams["figure.figsize"] = (7.2, 3.7)
fig, ax = plt.subplots(layout="constrained")
for date_text, color, marker, style, label in [
    ("2012-06-07", weekday_color, "s", "-", "Jun 7 (Thu)"),
    ("2012-06-09", weekend_color, "o", "--", "Jun 9 (Sat)"),
]:
    values = sample_hours.loc[sample_hours["dteday"] == date_text].sort_values("hr")
    ax.plot(values["hr"], values["cnt"], color=color, marker=marker,
            linestyle=style, markersize=3.4, label=label)
for hour in [8, 13, 17]:
    ax.axvline(hour, color="#9ba6af", linestyle=":", linewidth=0.8, zorder=0)
ax.set(xlabel="Clock hour", ylabel="Rentals per hour", xlim=(-0.3, 23.3),
       ylim=(0, 960), xticks=[0, 4, 8, 13, 17, 23])
ax.legend(frameon=False, loc="upper left")
plt.show()

**出力の見所：**日合計は7,494件と7,498件です。同じ横位置で8時台と13時台を比べ，どちらの日の件数が多いかが入れ替わることを説明してください。

### 2. 日付をまたぐ一時間

時計だけなら23時の次は0時です。日付も含めると，二つの時間帯はどの順番になるでしょうか。

In [ ]:
hourly["timestamp"] = hourly["dteday"] + pd.to_timedelta(hourly["hr"], unit="h")
hourly = hourly.sort_values("timestamp")
midnight = hourly["timestamp"].between("2012-06-07 23:00", "2012-06-08 00:00")
hourly.loc[midnight, ["dteday", "hr", "timestamp", "cnt"]]

### 3. 件数を足すか，行を数えるか

一日の貸出件数と，一日に収録された時間帯の数を別の列にします。10月29日の「1」と「22」は，それぞれ何を数えた値でしょうか。

In [ ]:
by_date = hourly.groupby("dteday")
rentals_per_day = by_date["cnt"].sum()
rows_per_day = by_date.size()

check_days = pd.DataFrame({
    "hours_in_table": rows_per_day,
    "rentals": rentals_per_day,
})
check_days.loc[[thursday, saturday, pd.Timestamp("2012-10-29")]]

In [ ]:
thursday_hours = hourly.loc[hourly["dteday"] == thursday]
time_block = thursday_hours["hr"] // 6
block_totals = thursday_hours.groupby(time_block)["cnt"].sum()
print(block_totals.to_list(), block_totals.sum())
block_totals

**出力の見所：**6月7日は24行を足して7,494件になります。6時間ごとの小計も同じ合計に戻りますが，8時や17時の山を個別に読むには元の時間別の図が必要です。

### 4. 同じ時刻の平均を作る

二日だけの違いでしょうか。24時刻の記録がそろう日を選び，土日・祝日と，それ以外の日を比べます。各時刻の平均は何日分から作られるか，先に確認します。

In [ ]:
rows_per_day = hourly.groupby("dteday").size()
complete_dates = rows_per_day.index[rows_per_day == 24]
complete_hours = hourly.loc[hourly["dteday"].isin(complete_dates)]
print("Days with 24 records:", len(complete_dates))
print(complete_hours.groupby("workingday")["dteday"].nunique())
same_hour = complete_hours.groupby(["workingday", "hr"])["cnt"]
mean_by_hour = same_hour.mean()
mean_by_hour.loc[[(0, 8), (1, 8), (0, 13), (1, 13)]].round(2)

In [ ]:
plt.rcParams["figure.figsize"] = (7.2, 3.7)
fig, ax = plt.subplots(layout="constrained")
ax.plot(mean_by_hour.loc[1].index, mean_by_hour.loc[1], color=weekday_color,
        marker="s", markersize=3, label="Weekday, excluding holidays")
ax.plot(mean_by_hour.loc[0].index, mean_by_hour.loc[0], color=weekend_color,
        marker="o", markersize=3, linestyle="--", label="Weekend or holiday")
for hour in [8, 13, 17]:
    ax.axvline(hour, color="#9ba6af", linestyle=":", linewidth=0.8, zorder=0)
ax.set(xlabel="Clock hour", ylabel="Mean rentals per hour", xlim=(-0.3, 23.3),
       ylim=(0, 630), xticks=[0, 4, 8, 13, 17, 23])
ax.legend(frameon=False, loc="upper left")
plt.show()

**出力の見所：**655日を使い，土日・祝日の各点は212日，それ以外の各点は443日を平均します。朝・昼・夕方で，同じ色の線がいつも高いかを読んでください。

### 5. 暦にあるのに表にない日

10月27日〜11月2日のうち，8〜18時台を選んで足します。7日間を選んだのに，集計結果は何行あるでしょうか。

In [ ]:
event_start = "2012-10-27"
event_end = "2012-11-02"
event_hours = hourly.loc[hourly["dteday"].between(event_start, event_end)]
daytime = event_hours.loc[event_hours["hr"].between(8, 18)]
observed_daytime = daytime.groupby("dteday")["cnt"].sum()
observed_daytime

In [ ]:
import matplotlib.dates as mdates
event_calendar = pd.date_range(event_start, event_end, freq="D")
daytime_on_calendar = observed_daytime.reindex(event_calendar)
daytime_on_calendar

In [ ]:
plt.rcParams["figure.figsize"] = (7.2, 5)
fig, axes = plt.subplots(2, 1, sharex=True, sharey=True, layout="constrained")
for ax, values, title in zip(
    axes, [observed_daytime, daytime_on_calendar],
    ["Dates present after grouping", "All calendar dates"],
):
    ax.plot(values.index, values, color=weekday_color, marker="o", markersize=4.5)
    ax.axvline(pd.Timestamp("2012-10-29"), color="#838e98", linestyle=":",
               linewidth=1.0)
    ax.set(title=title, ylabel="Rentals", ylim=(0, 6700))
axes[-1].xaxis.set_major_locator(mdates.DayLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
axes[-1].set_xlabel("Date (2012)")
plt.show()

**出力の見所：**10月29日には対象時間帯の行がなく，暦に合わせるとNaNになります。線がつながる図と切れる図で，29日の値について読み手が受け取る情報はどう違いますか。0件とみなす根拠は本文の元記録と出典で考えます。

### 6. 前日との違いを残す

隣り合う日の貸出件数の差と，日を一度並べ替えて隣にした差では，どちらが小さそうでしょうか。まず暦日に合わせ，前日の値を対応させます。

In [ ]:
all_days = pd.date_range(rentals_per_day.index.min(), rentals_per_day.index.max(), freq="D")
rentals_on_calendar = rentals_per_day.reindex(all_days)
print(len(rentals_on_calendar), rentals_on_calendar.isna().sum())

In [ ]:
previous_day = rentals_on_calendar.shift(1)
nearby = pd.DataFrame({"today": rentals_on_calendar, "yesterday": previous_day})
nearby["change"] = nearby["today"] - nearby["yesterday"]
nearby.loc["2012-10-28":"2012-10-31"]

In [ ]:
reordered = rentals_on_calendar.sample(frac=1, random_state=11)
day_gap = nearby["change"].abs()
reordered_gap = reordered.diff().abs()
week_gap = (rentals_on_calendar - rentals_on_calendar.shift(7)).abs()
print(day_gap.median(), reordered_gap.median(), week_gap.median())

**出力の見所：**前日との差の絶対値の中央値は468.5件，一度並べ替えた隣同士では1,808.5件です。値の集合は同じでも，どの日を隣に置くかで差は変わります。これは一度の順序比較で，置換検定を行ったものではありません。

## Jupyter 実習（約20分）

ここからは別の三つの水曜日を調べます。講義で使った6月7日・9日の比較から対象を替えます。

In [ ]:
wednesdays = pd.to_datetime(["2012-06-27", "2012-07-04", "2012-07-11"])
three_days = hourly.loc[hourly["dteday"].isin(wednesdays)]
three_days[["dteday", "hr", "cnt", "holiday"]].head()

## 1. 一日ごとの合計を見る（約 5 分）

時間別の表から日別の件数を作ります。`size()` で時刻行の数，`sum()` で貸出件数を調べます。`holiday` は D.C. の祝日の印です。

日別の結果を確認したら，**8 時台に最も貸出が多い日**を一つ予想し，理由も一文で書きます。

In [ ]:
daily_summary = three_days.groupby("dteday").agg(
    hours_in_table=("hr", "size"),
    rentals=("cnt", "sum"),
    holiday=("holiday", "first"),
)
daily_summary

（ここに予想と理由を書く）

## 2. 時刻別に比べる（約 7 分）

一日の中でいつ貸出が多いかを，同じ横軸と縦軸で比べます。横軸の `8` は 8 時台に始まった貸出です。三つの曲線で，朝が最も高い日と，20 時台以降が高い日を見分けてください。

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.5), layout="constrained")
for date, color, style in zip(
    wednesdays, ["#195b8a", "#a7421f", "#33735d"], ["-", "--", ":"]
):
    hours_for_date = three_days.loc[three_days["dteday"] == date].sort_values("hr")
    ax.plot(hours_for_date["hr"], hours_for_date["cnt"], color=color, linestyle=style,
            marker=".", label=date.strftime("%m-%d"))
ax.set(xlabel="Clock hour", ylabel="Rentals per hour", ylim=(0, 950),
       xlim=(-0.3, 23.3), xticks=[0, 4, 8, 12, 16, 20, 23])
ax.legend(frameon=False, loc="upper left")
plt.show()

three_days.loc[three_days["hr"].isin([8, 17, 21]), ["dteday", "hr", "cnt"]]

## 3. 調べる時間帯を変更する（約 8 分）

8〜10 時台をまとめた合計を出します。次に，セルの最初の二つの値を自分で変更して，20〜22 時台の合計を出してください。`between()` は両端を含むので，どちらの時間帯も 3 時刻分です。

三つの日を同じ順で表示するグラフです。**朝と夜でどの日が一番多かったか**，それぞれの合計とともに記録します。8 時台だけの比較と，8〜10 時台をまとめた比較で，最も多い日が同じかも確認します。最初の予想に戻り，日別・時間帯別の結果の両方を使って，朝の対応を考える人への説明を 2〜3 文で書きます。

In [ ]:
first_hour = 8
last_hour = 10

in_window = three_days["hr"].between(first_hour, last_hour)
window_totals = three_days.loc[in_window].groupby("dteday")["cnt"].sum().reindex(wednesdays)
window_totals.name = "rentals"

fig, ax = plt.subplots(figsize=(5.8, 3.0), layout="constrained")
ax.bar(window_totals.index.strftime("%m-%d"), window_totals,
       color=["#195b8a", "#a7421f", "#33735d"])
ax.set(xlabel="Date (2012)", ylabel="Rentals in selected hours",
       title=f"Clock hours {first_hour:02d}–{last_hour:02d}",
       ylim=(0, max(1800, 1.08 * window_totals.max())))
plt.show()

window_totals

（ここに朝と夜の比較，説明を書く）

### 理由を調べる（任意）

7 月 4 日は独立記念日です。国立公園局に，当日の夜の花火の記録があります。貸出の内訳を `cnt` から `registered` か `casual` へ変更し，同じ時間帯の比較で日付の順位がどう変わるか確かめます。夜の利用の理由をさらに調べるために必要な資料を一つ挙げてください。 [D.C. 人事局の 2012 年祝日表](https://dchr.dc.gov/sites/default/files/dc/sites/dchr/publication/attachments/DCHR_edpm_12_261_schedules_holidays_2012_2013.pdf) ・ [米国国立公園局の花火の記録](https://www.nps.gov/media/photo/gallery-item.htm?gid=AD5299B3-1DD8-B71B-0BADBF711BBDD03B&id=ad800bf9-1dd8-b71b-0b28-ff00da238951)

提出する場合はカーネルを再起動し，上から全セルを実行して，選んだ時間帯の図と説明が残る状態で保存します。

### 出典

H. Fanaee-T, *Bike Sharing*, UCI Machine Learning Repository（CC BY 4.0）。Capital Bikeshare の 2011〜2012 年の実測ログを研究者が集計した時間別の表を使いました。CSV は原配布の `hour.csv` の名前と改行コードを変更したものです。 [データ DOI](https://doi.org/10.24432/C5W894) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)